## Deteksi Anomali Transaksi Digital Berbasis Deep Learning
Implementasi arsitektur Neural Network (LSTM-Autoencoder) untuk mendeteksi pola penipuan (fraud) pada transaksi digital secara real-time dengan pendekatan *unsupervised learning*.

## 1. Inisialisasi & Simulasi Dataset (Data Engineering)

Kita tidak menggunakan dataset pasaran. Kita akan mensimulasikan data transaksi E-commerce dengan fitur: Amount, Time_Delta, IP_Change_Flag, Device_Score, dan Login_Frequency.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, embedding_dim=64):
        super(LSTMAutoencoder, self).__init__()
        self.seq_len = seq_len
        self.n_features = n_features

        # Encoder
        self.encoder = nn.LSTM(n_features, embedding_dim, batch_first=True)

        # Decoder
        self.decoder = nn.LSTM(embedding_dim, n_features, batch_first=True)

    def forward(self, x):
        # Step 1: Encode ke hidden state terakhir
        _, (hidden, _) = self.encoder(x)

        # Step 2: Repeat hidden state untuk input decoder
        # (batch, seq_len, embedding_dim)
        x_inv = hidden[-1].repeat(self.seq_len, 1, 1).permute(1, 0, 2)

        # Step 3: Decode kembali ke dimensi fitur asli
        x_recon, _ = self.decoder(x_inv)
        return x_recon

## 2. Arsitektur Model: LSTM-Autoencoder

Model ini bekerja dengan prinsip Reconstruction Error. Model ini belajar "mengompresi" 10 transaksi menjadi satu vektor kecil, lalu mencoba "menebak ulang" (rekonstruksi) transaksi tersebut.

In [ ]:
def generate_data(n_users, is_fraud=False):
    """Menghasilkan urutan transaksi sintetis."""
    data = torch.randn(n_users, SEQ_LEN, FEATURES) * 0.1
    if is_fraud:
        # Penipuan: Anomali pada fitur 0 (Amount) dan fitur 2 (IP Change) di akhir urutan
        data[:, -1, 0] += 5.0
        data[:, -1, 2] += 3.0
    return data

# Konfigurasi Dasar
SEQ_LEN = 10    # 10 transaksi terakhir per user
FEATURES = 5    # Jumlah fitur per transaksi
BATCH_SIZE = 32

model = LSTMAutoencoder(SEQ_LEN, FEATURES)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Buat data latihan (Hanya data Normal)
train_data = generate_data(1000, is_fraud=False)
train_loader = DataLoader(TensorDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)

# Buat data uji (Campuran)
test_normal = generate_data(100, is_fraud=False)
test_fraud = generate_data(20, is_fraud=True)

print(f"Data Training: {train_data.shape}")
print(f"Data Uji Normal: {test_normal.shape}")
print(f"Data Uji Fraud: {test_fraud.shape}")

### Explanatory Data Analysis

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Konversi tensor ke DataFrame untuk EDA
# Kita ambil transaksi terakhir (index -1) dari setiap sekuens
normal_last = test_normal[:, -1, :].numpy()
fraud_last = test_fraud[:, -1, :].numpy()

cols = ['Amount', 'Time_Delta', 'IP_Change', 'Device_Score', 'Login_Freq']
df_normal = pd.DataFrame(normal_last, columns=cols)
df_normal['Label'] = 'Normal'
df_fraud = pd.DataFrame(fraud_last, columns=cols)
df_fraud['Label'] = 'Fraud'

df_combined = pd.concat([df_normal, df_fraud])

# Visualisasi Distribusi Amount (Fitur Paling Kritikal)
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_combined, x='Amount', hue='Label', fill=True)
plt.title("Distribusi Nilai Transaksi: Normal vs Fraud")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def plot_temporal_velocity(test_normal, test_fraud, user_idx=0):
    # Ambil data fitur 'Amount' (index 0) dan 'Time_Delta' (index 1)
    normal_user = test_normal[user_idx, :, :2].numpy()
    fraud_user = test_fraud[user_idx, :, :2].numpy()

    fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Plot User Normal
    ax[0].plot(normal_user[:, 0], marker='o', label='Amount', color='blue')
    ax[0].set_title("Pola Transaksi Normal (Stabil)")
    ax[0].set_ylabel("Normalized Amount")
    ax[0].grid(True, linestyle='--', alpha=0.6)

    # Plot User Fraud (Carding Pattern)
    ax[1].plot(fraud_user[:, 0], marker='s', label='Amount', color='red')
    ax[1].set_title("Pola Transaksi Fraud (Exponential/Carding)")
    ax[1].set_ylabel("Normalized Amount")
    ax[1].set_xlabel("Urutan Transaksi (T-9 sampai T-0)")
    ax[1].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

plot_temporal_velocity(test_normal, test_fraud)

In [ ]:
import seaborn as sns

def plot_correlation_comparison(df_normal, df_fraud):
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    # Korelasi pada data Normal
    sns.heatmap(df_normal.corr(), annot=True, cmap='coolwarm', ax=ax[0], fmt='.2f')
    ax[0].set_title("Korelasi Fitur: Kondisi Normal")

    # Korelasi pada data Fraud
    sns.heatmap(df_fraud.corr(), annot=True, cmap='coolwarm', ax=ax[1], fmt='.2f')
    ax[1].set_title("Korelasi Fitur: Kondisi Fraud (Sinyal Serangan)")

    plt.tight_layout()
    plt.show()

# Menggunakan dataframe dari langkah EDA sebelumnya
plot_correlation_comparison(df_normal.drop('Label', axis=1), df_fraud.drop('Label', axis=1))

# 3. Fase Pelatihan (Model Learning)

Model hanya dilatih pada data Normal. Tujuannya agar model sangat mahir merekonstruksi pola belanja yang wajar.

In [ ]:
EPOCHS = 30
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_loader:
        inputs = batch[0]
        outputs = model(inputs)
        loss = criterion(outputs, inputs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.6f}")

## 4. Evaluasi & Ambang Batas (Anomaly Thresholding)

Kita hitung seberapa besar kesalahan rekonstruksi (Reconstruction Error). Jika error besar, berarti model "bingung" melihat transaksi tersebut—indikasi kuat penipuan.

In [ ]:
model.eval()

def calculate_anomaly_score(data):
    with torch.no_grad():
        recons = model(data)
        # Hitung MSE per individu (bukan rata-rata batch)
        return torch.mean((recons - data)**2, dim=(1, 2))

loss_normal = calculate_anomaly_score(test_normal)
loss_fraud = calculate_anomaly_score(test_fraud)

# Menentukan Threshold (Gunakan persentil 95 dari data normal)
threshold = np.percentile(loss_normal.numpy(), 95)
print(f"Ambang Batas (Threshold): {threshold:.4f}")

## 5. Adaptive Thresholding & Penanganan Concept Drift
Dalam skenario dunia nyata, perilaku belanja bersifat dinamis (misal: lonjakan transaksi pada musim liburan).
Implementasi **Exponential Moving Average (EMA)** pada *threshold* memungkinkan sistem untuk:
1. **Self-Adjusting:** Mengikuti tren fluktuasi error tanpa perlu intervensi manual.
2. **Robustness:** Mengurangi angka *False Positive* pada periode trafik tinggi.

In [ ]:
import torch

class DynamicThreshold:
    """
    Menggunakan Exponential Moving Average (EMA) untuk menyesuaikan ambang batas
    secara real-time berdasarkan fluktuasi perilaku transaksi normal.
    """
    def __init__(self, alpha=0.1, initial_threshold=0.05):
        self.alpha = alpha
        self.current_threshold = initial_threshold
        self.history = [initial_threshold]

    def update(self, new_batch_losses):
        # Kita hanya mengupdate threshold berdasarkan transaksi yang DIKONFIRMASI normal
        # batch_mean dikali 3 sebagai buffer standar (3-sigma rule)
        batch_mean = torch.mean(new_batch_losses).item()

        # Rumus EMA: New = (1 - alpha) * Old + alpha * Current_Observation
        self.current_threshold = (1 - self.alpha) * self.current_threshold + self.alpha * (batch_mean * 3)
        self.history.append(self.current_threshold)
        return self.current_threshold

# 1. Inisialisasi dengan threshold awal (dari persentil 95 data normal awal)
dy_threshold = DynamicThreshold(alpha=0.05, initial_threshold=threshold)

# 2. Asumsikan ada batch transaksi baru yang masuk dan dikonfirmasi normal oleh sistem
# Kita update threshold agar sistem 'belajar' bahwa saat ini sedang ada kenaikan trafik normal
new_threshold = dy_threshold.update(loss_normal[:10])

print(f"Threshold Awal: {threshold:.4f}")
print(f"Threshold Adaptif Terbaru: {new_threshold:.4f}")

## 6. Logika Keputusan & Nilai Jual Keamanan

Dalam sistem nyata, kodenya tidak berhenti pada deteksi, tapi pada aksi:

In [ ]:
def real_time_decision(loss_value, threshold):
    if loss_value > threshold * 3:
        return "BLOCK & BAN"  # Sangat mencurigakan
    elif loss_value > threshold:
        return "TRIGGER OTP"  # Zona abu-abu, butuh verifikasi tambahan
    else:
        return "APPROVE"      # Transaksi aman

# Simulasi pada satu data fraud
sample_loss = loss_fraud[0].item()
print(f"Loss: {sample_loss:.4f} -> Action: {real_time_decision(sample_loss, threshold)}")

SHAP biasanya digunakan untuk model klasifikasi atau regresi standar. Karena model ini adalah Autoencoder, "output" yang ingin dijelaskan adalah Mean Squared Error (MSE).

Fungsi ini merubah data dari format Numpy (yang diminta SHAP) ke Tensor (yang diminta PyTorch), menghitung error-nya, lalu mengembalikannya ke Numpy.

In [ ]:
import shap
import torch

# 1. Definisikan fungsi yang mengembalikan MSE per sampel
def predict_loss(data_np):
    model.eval()
    # Konversi kembali dari numpy (format SHAP) ke torch tensor
    data_torch = torch.FloatTensor(data_np).view(-1, SEQ_LEN, FEATURES)
    with torch.no_grad():
        reconstructed = model(data_torch)
        # Hitung MSE untuk setiap sampel dalam batch
        # SHAP berekspektasi output 1D (skor anomali)
        loss = torch.mean((data_torch - reconstructed)**2, dim=(1, 2))
    return loss.numpy()

# 2. Siapkan data untuk SHAP
# Background: Data normal sebagai referensi "ekspektasi" model
background = train_data[:100].view(-1, SEQ_LEN * FEATURES).numpy()
# Test Sample: Satu sampel fraud yang ingin kita bongkar alasannya
test_sample = test_fraud[0:1].view(-1, SEQ_LEN * FEATURES).numpy()

# 3. Gunakan KernelExplainer (paling fleksibel untuk PyTorch)
# Kita bungkus fungsi agar menerima data dalam bentuk flat (n_samples, seq*features)
def wrapper_func(x_flat):
    return predict_loss(x_flat)

explainer = shap.KernelExplainer(wrapper_func, background)
shap_values = explainer.shap_values(test_sample)

`shap.KernelExplainer` membandingkan satu transaksi mencurigakan (test_sample) dengan sekumpulan transaksi normal (background). Ia akan melihat fitur mana yang paling bertanggung jawab menyebabkan lonjakan loss.

In [ ]:
# Nama fitur untuk 10 urutan transaksi (T-9 sampai T-0)
feature_names = []
for i in range(SEQ_LEN):
    for f in ['Amount', 'Time', 'IP', 'Device', 'Freq']:
        feature_names.append(f"T-{9-i}_{f}")

# Visualisasi kontribusi fitur
shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    test_sample[0],
    feature_names=feature_names,
    matplotlib=True
)

## 7. Adversarial Attack Simulation (Uji Ketahanan Model)
Adversarial Fraud adalah transaksi penipuan yang dibuat sangat halus agar Reconstruction Error-nya rendah dan lolos dari deteksi.

In [ ]:
def generate_adversarial_fraud(n_samples, model, epsilon=0.1):
    model.eval()
    data = generate_data(n_samples, is_fraud=True)
    data.requires_grad = True

    # Hitung gradien loss terhadap input
    output = model(data)
    loss = criterion(output, data)
    model.zero_grad()
    loss.backward()

    # Ubah data fraud agar error-nya mengecil (seolah-olah normal)
    with torch.no_grad():
        adv_data = data - epsilon * data.grad.sign()
    return adv_data

adv_samples = generate_adversarial_fraud(10, model)
adv_loss = calculate_anomaly_score(adv_samples)

print(f"Rata-rata Loss Fraud Biasa: {loss_fraud.mean().item():.4f}")
print(f"Rata-rata Loss Adv Fraud (Penipu Pintar): {adv_loss.mean().item():.4f}")

## 8. Analisis Performa Bisnis (Precision-Recall)

Deteksi penipuan adalah masalah klasifikasi yang sangat tidak seimbang (*imbalanced*). Oleh karena itu, kita menggunakan **Precision-Recall Curve** sebagai metrik utama, bukan Accuracy.

**Kesimpulan Evaluasi:**
* **PR AUC Score:** Menunjukkan kemampuan agregat model dalam mengidentifikasi fraud di berbagai tingkatan sensitivitas.
* **F1-Score Optimization:** Kita menemukan *Threshold* optimal sebesar **[Isi Hasil Anda]** yang menyeimbangkan antara deteksi fraud dan minimalisasi *False Positive*.

**Rekomendasi Operasional:**
Sistem akan menggunakan *threshold* dinamis yang diinisialisasi dari titik optimal F1-score ini, kemudian disesuaikan secara adaptif menggunakan algoritma EMA yang telah diimplementasikan sebelumnya.

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score, auc

# 1. Gabungkan hasil untuk evaluasi (0 = Normal, 1 = Fraud)
y_true = np.concatenate([np.zeros(len(loss_normal)), np.ones(len(loss_fraud))])
y_scores = np.concatenate([loss_normal.numpy(), loss_fraud.numpy()])

# 2. Hitung Precision-Recall
precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall, precision)

# 3. Visualisasi
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='purple', lw=2, label=f'PR Curve (AUC = {pr_auc:.2f})')
plt.fill_between(recall, precision, alpha=0.2, color='purple')
plt.xlabel('Recall (Kemampuan Menangkap Fraud)')
plt.ylabel('Precision (Keakuratan Blokir)')
plt.title('Precision-Recall Curve: Analisis Trade-off Keamanan')
plt.legend(loc="lower left")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 4. Mencari Threshold Optimal (F1-Score Tertinggi)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]

print(f"Hasil Evaluasi:")
print(f"--------------------------")
print(f"PR AUC Score: {pr_auc:.4f}")
print(f"Best F1-Score: {f1_scores[best_idx]:.4f}")
print(f"Recommended Threshold (Max F1): {best_threshold:.4f}")